# Chapter 3: Statsmodels — Regression With Full Statistical Inference

### Why This Matters

Volume 05 will introduce scikit-learn, which is optimized for **prediction** — "what will sales be next month?" — and mostly treats a model's internals as a black box you evaluate purely by predictive accuracy. **Statsmodels** takes the opposite emphasis: it's built for **inference** — "*why* did sales change, by how much, attributable to which factor, and how confident can I be in that estimate?" As an analyst whose job frequently involves explaining and defending conclusions to stakeholders (not just generating a prediction number), this inference-first tool belongs in your kit alongside scikit-learn, not instead of it.

### 3.1 OLS Regression With Full Interpretation

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

df = pd.DataFrame({
    'sales': [120, 140, 155, 170, 190, 210, 230],
    'ad_spend': [10, 15, 18, 22, 25, 30, 35]
})

model = smf.ols('sales ~ ad_spend', data=df).fit()
print(model.summary())

The `'sales ~ ad_spend'` syntax (called a **formula**, borrowed from the R statistical language) reads naturally as "model sales as a function of ad_spend." `.fit()` estimates the model, and `.summary()` prints a dense but extremely information-rich report. The parts worth learning to read, in order of practical importance:

- **`coef` (coefficient) on `ad_spend`** — the estimated change in `sales` for every 1-unit increase in `ad_spend`, holding everything else in the model constant. This is the headline business number: "every additional ₹1 lakh of ad spend is associated with an estimated ₹X lakh increase in sales."
- **`P>|t|` (p-value) for that coefficient** — is this relationship statistically distinguishable from zero (no relationship), given the amount of data you have? Below 0.05 conventionally counts as "significant."
- **`[0.025  0.975]` (the 95% confidence interval)** — the plausible range for the true coefficient. A business-friendly way to phrase this: "we estimate each additional lakh of ad spend drives somewhere between X and Y lakhs of additional sales, with 95% confidence" — always more honest than reporting a single point estimate as if it were exact.
- **`R-squared`** — the proportion of variation in `sales` this model explains, from 0 (explains nothing) to 1 (perfectly explains it). A useful headline number, but never sufficient on its own — a high R² with a nonsensical or non-causal predictor is still a bad model, and a modest R² can still represent a genuinely useful, statistically solid finding in noisy real-world business data.

In [ ]:
print(model.params)        # just the coefficients, no formatting
print(model.conf_int())    # just the confidence intervals

### 3.2 Multiple Regression and Confounding

In [ ]:
df['team_size'] = [5, 6, 6, 7, 8, 9, 10]

model2 = smf.ols('sales ~ ad_spend + team_size', data=df).fit()
print(model2.summary())

Adding a second predictor changes the interpretation of the first in a way that matters enormously in practice: the `ad_spend` coefficient in `model2` now represents the effect of ad spend **holding team size constant** — which can be meaningfully different (sometimes smaller, sometimes larger, occasionally even flipping sign) from its coefficient in the single-predictor `model`. This is the practical face of **confounding**: if `ad_spend` and `team_size` tend to rise together (a growing company hires more people *and* spends more on ads simultaneously), a single-predictor model risks crediting `ad_spend` alone with an effect that's really shared with (or driven by) team growth. This is exactly the statistical machinery behind the "correlation ≠ causation" warning from Chapter 2 — multiple regression is one of the standard tools for trying to isolate one factor's effect from others it's entangled with, though even it can't fully substitute for a proper controlled experiment.

### 3.3 Logistic Regression — When the Outcome Is Categorical

In [ ]:
df_churn = pd.DataFrame({
    'churned': [0, 0, 1, 0, 1, 1, 0, 1, 0, 1],          # 0 = stayed, 1 = churned
    'tenure_months': [24, 36, 3, 18, 2, 5, 30, 4, 22, 6],
    'monthly_spend': [800, 950, 400, 700, 350, 450, 900, 380, 750, 420]
})

logit_model = smf.logit('churned ~ tenure_months + monthly_spend', data=df_churn).fit()
print(logit_model.summary())
print("\nOdds ratios:\n", np.exp(logit_model.params))

**OLS regression** (Section 3.1) predicts a continuous number (sales in rupees); **logistic regression** predicts the probability of a binary categorical outcome (churned or not, converted or not, defaulted or not) — precisely the churn-prediction problem this course's capstone (Volume 11) revisits with scikit-learn's implementation. The coefficients themselves are on a harder-to-interpret "log-odds" scale, which is why `np.exp(coefficient)` (converting to an **odds ratio**) is the standard way to make them business-readable: an odds ratio of 1.5 on a variable means "each 1-unit increase in that variable is associated with 1.5x the odds of churning," while an odds ratio below 1 means the variable is *protective* against the outcome.

### 3.4 Time Series: A First Look

In [ ]:
sales_ts = pd.Series([120, 135, 128, 160, 172, 155, 180, 190, 205, 198, 215, 225],
                       index=pd.date_range('2025-01-01', periods=12, freq='MS'))

# Decompose into trend, seasonality, and residual noise
from statsmodels.tsa.seasonal import seasonal_decompose
# decomposition = seasonal_decompose(sales_ts, model='additive', period=4)   # needs >= 2 full periods of data
# decomposition.plot()

# A simple, classic forecasting baseline: ARIMA
from statsmodels.tsa.arima.model import ARIMA
arima_model = ARIMA(sales_ts, order=(1, 1, 1)).fit()      # (p, d, q) — AR order, differencing, MA order
forecast = arima_model.forecast(steps=3)
print("Next 3 months forecast:\n", forecast)

Time series analysis deserves an entire course of its own, but the essential vocabulary worth knowing: **trend** (the long-run direction), **seasonality** (a repeating pattern tied to a fixed period — weekly, monthly, quarterly), and **residual/noise** (whatever's left after removing trend and seasonality) are the three components `seasonal_decompose` splits any time series into. **ARIMA** (AutoRegressive Integrated Moving Average) is a classic, still widely-used forecasting model, configured by three numbers `(p, d, q)`: how many past values to regress on (`p`), how many times to difference the series to make it stationary/stable (`d`), and how many past forecast errors to incorporate (`q`). Choosing good `(p, d, q)` values is itself a small discipline (examining autocorrelation plots, or using automated tools like `pmdarima`'s `auto_arima`) — treat the values above as illustrative rather than something to blindly reuse.

### 3.5 Statsmodels vs. Scikit-learn — When to Use Which

| | Statsmodels | Scikit-learn (Volume 05) |
|---|---|---|
| Primary goal | Inference — understand and explain relationships | Prediction — maximize accuracy on new data |
| Output | Full statistical summary: coefficients, p-values, confidence intervals | A fitted model object; accuracy metrics on a test set |
| Best for | "Why did this happen, and how confident are we?" | "What will happen next?" |
| Typical audience | Analysts, economists, researchers defending a conclusion | ML engineers optimizing a deployed prediction system |

**In practice, professional analysts use both**, often on the same problem: scikit-learn to build the best-performing predictive model, Statsmodels to run the equivalent regression and pull defensible statistical language ("this factor's effect is statistically significant, with a 95% confidence interval of...") for the accompanying business narrative.

### Cheat Sheet — Statsmodels

| Task | Code |
|---|---|
| OLS regression | `smf.ols('y ~ x1 + x2', data=df).fit()` |
| View full report | `model.summary()` |
| Coefficients / CIs alone | `model.params`, `model.conf_int()` |
| Logistic regression | `smf.logit('y ~ x1 + x2', data=df).fit()` |
| Odds ratios | `np.exp(logit_model.params)` |
| Time series decomposition | `seasonal_decompose(series, model='additive', period=N)` |
| ARIMA forecast | `ARIMA(series, order=(p,d,q)).fit().forecast(steps=N)` |

### 🎯 Business Challenge — Statsmodels

*Task (adapted from the source material's mini project):* analyze sales drivers using multiple regression with a full executive interpretation.

In [ ]:
np.random.seed(3)
biz = pd.DataFrame({
    "ad_spend": np.random.uniform(10, 60, 40),
    "team_size": np.random.randint(4, 20, 40),
    "region_tier": np.random.choice([1, 2, 3], 40)
})
biz["sales"] = (8*biz["ad_spend"] + 12*biz["team_size"] - 5*biz["region_tier"]
                + np.random.normal(0, 15, 40) + 50)

model = smf.ols('sales ~ ad_spend + team_size + region_tier', data=biz).fit()
print(model.summary())

print(f"""
EXECUTIVE INTERPRETATION
------------------------
Every additional ₹1 lakh of ad spend is associated with an estimated
{model.params['ad_spend']:.1f} lakh increase in sales (p={model.pvalues['ad_spend']:.4f}),
holding team size and region tier constant.
The model explains {model.rsquared*100:.1f}% of the variation in sales (R-squared).
""")

---

## Volume 04 — What You Should Be Able to Do Now

- Choose mean vs. median appropriately based on skew, and explain why to a non-technical audience
- Use the IQR rule to flag statistical outliers and articulate what to do next (investigate, don't automatically delete)
- Run and interpret a t-test to decide whether an observed difference between two groups is likely real
- Fit an OLS regression with Statsmodels and correctly interpret the coefficient, p-value, confidence interval, and R²
- Explain, in plain language, the difference between what Statsmodels and scikit-learn are each optimized for

**Next: `05_Machine_Learning_Foundations.md`**